# 05 — Alternative Global Models (C1–C2)

This notebook trains two additional global models:

| ID | Model | Notes |
|----|-------|-------|
| C1 | Global LightGBM | Gradient boosting with early stopping |
| C2 | Global Ridge | Linear baseline, fast |

All training uses 5-fold CV on train split only. Test data is used once at the end for evaluation.

In [1]:
import json
import os, time, warnings
from pathlib import Path

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.dirname(os.getcwd()))

import sys
if "src" not in sys.path[0]:
    sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import joblib
warnings.filterwarnings("ignore")

from creator_analytics import features as fe
from creator_analytics.config import load_config
from creator_analytics.models import (
    VS_CREATOR_SUFFIX, prepare_X,
    search_lightgbm, train_ridge,
    TrainedModel,
)
from creator_analytics.evaluate import (
    compute_metrics, compute_per_creator_metrics,
    paired_comparison, log_experiment,
)

cfg       = load_config("config/default.yaml")
PROCESSED = Path(cfg["paths"]["processed_dir"])
PREDS_DIR = Path("results/predictions"); PREDS_DIR.mkdir(parents=True, exist_ok=True)
FIGS      = Path("results/figures"); FIGS.mkdir(parents=True, exist_ok=True)
SEED      = cfg["seed"]
TODAY     = "2026-10-25"


In [2]:
data, spec = fe.load_feature_table(PROCESSED)
idx  = pd.read_parquet(PROCESSED / "dataset_index.parquet")

TRAIN = idx["split"].eq("train")
TEST  = idx["split"].eq("test")

FEATURES_GLOBAL = spec["features"]

X_all_global  = data[FEATURES_GLOBAL].copy()
y_all  = idx["target_rel"].copy()
auth   = idx["author_id"].copy()

X_train_global = X_all_global[TRAIN.values].reset_index(drop=True)
y_train        = y_all[TRAIN.values].reset_index(drop=True)
auth_train     = auth[TRAIN.values].reset_index(drop=True)

X_test_global  = X_all_global[TEST.values].reset_index(drop=True)
y_test         = y_all[TEST.values].reset_index(drop=True)
auth_test      = auth[TEST.values].reset_index(drop=True)

print(f"Global features: {len(FEATURES_GLOBAL)}")
print(f"Train: {len(X_train_global):,}  Test: {len(X_test_global):,}")


Global features: 106
Train: 160,492  Test: 40,124


In [3]:
# Load B2 baseline from NB04
b2_model = TrainedModel.load("models/B2_global_xgb.joblib")
nb04     = pd.read_parquet(PREDS_DIR / "nb04_predictions.parquet")
b2_preds = nb04["B2_XGB"].values
print("B2 baseline RMSE:", round(compute_metrics(y_test, b2_preds)["rmse"], 4))


B2 baseline RMSE: 0.9127


## C1 — Global LightGBM

In [4]:
ckpt      = Path("models/alternative/C1_global_lgbm.joblib")
meta_path = ckpt.with_name(ckpt.stem + "_meta.json")
if ckpt.exists():
    lgbm_global = TrainedModel.load(ckpt)
    elapsed_c1g = json.loads(meta_path.read_text())["elapsed_s"] if meta_path.exists() else None
    t_str = f"time={elapsed_c1g/60:.1f}min" if elapsed_c1g else "(loaded)"
    print(f"C1 global LGBM  CV RMSE={lgbm_global.cv_rmse:.4f}  {t_str}")
else:
    t0 = time.time()
    lgbm_global = search_lightgbm(
        X_train_global, y_train, auth_train,
        n_splits=cfg["cv"]["n_splits"], seed=SEED, n_iter=30,
    )
    elapsed_c1g = time.time() - t0
    lgbm_global.save(ckpt)
    meta_path.write_text(json.dumps(
        {"elapsed_s": elapsed_c1g, "cv_rmse": lgbm_global.cv_rmse, "params": lgbm_global.params}, indent=1))
    print(f"C1 global LGBM  CV RMSE={lgbm_global.cv_rmse:.4f}  time={elapsed_c1g/60:.1f}min")


C1 global LGBM  CV RMSE=0.9135  (loaded)


## C2 — Global Ridge

In [5]:
ckpt      = Path("models/alternative/C2_global_ridge.joblib")
meta_path = ckpt.with_name(ckpt.stem + "_meta.json")
if ckpt.exists():
    ridge_global = TrainedModel.load(ckpt)
    elapsed_c2g = json.loads(meta_path.read_text())["elapsed_s"] if meta_path.exists() else None
    t_str = f"time={elapsed_c2g:.1f}s" if elapsed_c2g else "(loaded)"
    print(f"C2 global Ridge  CV RMSE={ridge_global.cv_rmse:.4f}  alpha={ridge_global.params.get('alpha')}  {t_str}")
else:
    t0 = time.time()
    ridge_global = train_ridge(
        X_train_global, y_train, auth_train,
        n_splits=cfg["cv"]["n_splits"], seed=SEED,
    )
    elapsed_c2g = time.time() - t0
    ridge_global.save(ckpt)
    meta_path.write_text(json.dumps(
        {"elapsed_s": elapsed_c2g, "cv_rmse": ridge_global.cv_rmse,
         "alpha": ridge_global.params.get("alpha")}, indent=1))
    print(f"C2 global Ridge  CV RMSE={ridge_global.cv_rmse:.4f}  alpha={ridge_global.params['alpha']}  time={elapsed_c2g:.1f}s")


C2 global Ridge  CV RMSE=0.9674  alpha=100.0  (loaded)


## Evaluation on held-out test set

In [6]:
all_preds = {
    "B2_global_XGB":   b2_preds,
    "C1_global_LGBM":  lgbm_global.predict(X_test_global),
    "C2_global_Ridge": ridge_global.predict(X_test_global),
}

metrics = {n: compute_metrics(y_test, p) for n, p in all_preds.items()}
df_m = pd.DataFrame(metrics).T[["rmse", "mae", "r2", "spearman", "n"]]
display(df_m.round(4))


,rmse,mae,r2,spearman,n
B2_global_XGB,0.9127,0.5855,0.1499,0.3101,40124.0
C1_global_LGBM,0.9104,0.5841,0.1541,0.3198,40124.0
C2_global_Ridge,0.9718,0.6182,0.0362,0.1416,40124.0


In [7]:
# Save predictions
df_out = pd.DataFrame({
    "video_id":  idx.loc[TEST, "video_id"].values,
    "author_id": auth_test.values,
    "y_true":    y_test.values,
})
for name, preds in all_preds.items():
    df_out[name] = preds
df_out.to_parquet(PREDS_DIR / "nb05_predictions.parquet", index=False)

# Save models
Path("models/hybrid").mkdir(parents=True, exist_ok=True)
lgbm_global.save("models/alternative/C1_global_lgbm.joblib")
ridge_global.save("models/alternative/C2_global_ridge.joblib")

# Save training times for NB07
times_path = Path("models/alternative/training_times.json")
times = json.loads(times_path.read_text()) if times_path.exists() else {}
times["C1_global_LGBM"]  = elapsed_c1g
times["C2_global_Ridge"] = elapsed_c2g
json.dump(times, open(times_path, "w"), indent=1)
print("Saved predictions, models, training times")


Saved predictions, models, training times


In [8]:
# Log experiments
exp_path = Path(cfg["paths"]["experiments_log"])
for name, preds in all_preds.items():
    if name == "B2_global_XGB":
        continue
    cmp_b2 = paired_comparison(y_test, preds, b2_preds, auth_test, name, "B2_XGB")
    log_experiment(
        exp_path, name=name, date=TODAY,
        features=f"{len(FEATURES_GLOBAL)} global",
        model=name, params={}, cv_rmse=None,
        test_metrics=compute_metrics(y_test, preds),
        train_time_s={"C1_global_LGBM": elapsed_c1g, "C2_global_Ridge": elapsed_c2g}.get(name),
        comparison=cmp_b2,
        comment="NB05; vs B2 global XGB",
    )
print("Logged experiments")


Logged experiments


## Summary

*(Filled from cell output)*

**Key findings:**
- C1 global LGBM vs B2 global XGB: comparable gradient-boosting performance
- C2 global Ridge: linear baseline, interpretable, fast